# TP02 — OOP
## Exercise 1: Build the `CSVReader` Class

### Objective
This exercise practices the basic concepts of Object-Oriented Programming (OOP):

- **Class** A class is a template for creating objects.
- **Constructor (`__init__`)** A constructor is the special method that runs automatically when an object is created.
- **Methods** A method is a function that belongs to a class.
- **Encapsulation** Encapsulation means keeping related data and functions together inside one class.

The `CSVReader` class will keep all CSV-reading operations inside one reusable class.

In [1]:
import pandas as pd

## 2 — Create the `CSVReader` Class

The class contains:

- `__init__(file_path)` → stores the CSV file path. A constructor is a special method that runs automatically when you create a new object.
- `read()` → reads the CSV file using pandas.
- `preview(n)` → prints the first `n` rows.

`self.data` stores the loaded DataFrame so that other methods can reuse it.

In [2]:

class CSVReader:
    """A simple reusable class for reading and previewing CSV files."""

    def __init__(self, file_path):
        """Store the path of the CSV file."""
        self.file_path = file_path
        self.data = None

    def read(self):
        """Read the CSV file and store it in self.data."""
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        """Print the first n rows of the loaded CSV file."""
        if self.data is None:
            self.read()

        print(self.data.head(n))


In [5]:
reader = CSVReader("D:\I5\ADV\sample_data (2).csv")
df = reader.read()
df

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


## 5 — Preview the First 5 Rows

The `preview()` method displays the top rows of the dataset.  
By default, `n=5`.

In [6]:
reader.preview(5) 

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## 6 — Preview a Different Number of Rows

Because `preview(n)` accepts an argument, we can choose how many rows to display.

In [7]:
reader.preview(3)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN


## Result

The `CSVReader` class successfully:

1. Stores the CSV file path.
2. Reads the file with pandas.
3. Stores the DataFrame inside the object.
4. Previews any requested number of rows.

This completes **Exercise 1**.

# Exercise 2: Apply Pattern for Missing Values Handling

## Objective

- Learn the Strategy Pattern: one interface, many interchangeable behaviors.
- Practice inheritance and polymorphism.

## Concept

Different strategies can be used to handle missing data, such as dropping missing values, filling them with the mean, or filling them with the mode.

The Strategy Pattern allows us to switch between these methods without changing the main code.

## Instructions:
1. Create an abstract class MissingValueStrategy with a method handle(df).
2. Create subclasses:
- oDropMissing
- oFillMean
- oFillMode
3. Create a DataCleaner class that applies the chosen strategy.

In [8]:
# 1. Create an abstract class MissingValueStrategy with a method handle(df).
from abc import ABC, abstractmethod


class MissingValueStrategy(ABC):

    @abstractmethod
    def handle(self, df):
        pass

In [9]:
# 2. Create subclasses:
# DropMissing
class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna()

In [ ]:


# 1. Create a class to remove missing values
class DropMissing:
    def clean(self, df):
        return df.dropna()

# 2. Create the DataCleaner class
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.clean(df)

# 3. Load the CSV file
df = pd.read_csv(r"D:\I5\ADV\sample_data (2).csv")

# 4. Create the cleaner
cleaner = DataCleaner(DropMissing())

# 5. Clean the data
cleaned_df = cleaner.clean(df)

# 6. Display the first five rows
print(cleaned_df.head())

# 7. Check the number of rows before and after cleaning
print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)

   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0
Original shape: (7, 7)
Cleaned shape: (2, 7)


In [17]:
# FillMean
class FillMean(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()
        numeric_cols = result.select_dtypes(include="number").columns
        result[numeric_cols] = result[numeric_cols].fillna(
            result[numeric_cols].mean()
        )
        return result

In [19]:
# FillMode
class FillMode(MissingValueStrategy):
    def handle(self, df):
        result = df.copy()

        for column in result.columns:
            if result[column].isna().any():
                mode_values = result[column].mode()

                if not mode_values.empty:
                    result[column] = result[column].fillna(
                        mode_values.iloc[0]
                    )

        return result

In [26]:
class FillMode:
    def clean(self, df):
        df = df.copy()

        for column in df.columns:
            if df[column].isnull().any():
                mode_values = df[column].mode()

                if not mode_values.empty:
                    df[column] = df[column].fillna(mode_values.iloc[0])

        return df

In [24]:
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.clean(df)

In [27]:
import pandas as pd

df = pd.read_csv(r"D:\I5\ADV\sample_data (2).csv")

cleaner = DataCleaner(FillMode())
cleaned_df = cleaner.clean(df)

print(cleaned_df.head())

print("Missing values before cleaning:")
print(df.isnull().sum())

print("Missing values after cleaning:")
print(cleaned_df.isnull().sum())

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3   4    David  27.0      160.0       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
Missing values before cleaning:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64
Missing values after cleaning:
id           0
name         0
age          0
height_cm    0
weight_kg    0
city         0
score        0
dtype: int64


In [28]:
# 3. Create a DataCleaner class that applies the chosen strategy.
class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

# Exercise 3: Add Decorators for Logging and Timing

**Objectives**
- Learn the **Decorator Pattern** to add reusable behaviour (logging, timing).
- Understand how decorators support the **Open/Closed Principle**: classes stay *closed for modification* but *open for extension*.

**Concept:** a decorator is a function that takes a function, wraps it with extra behaviour, and returns the wrapper. The original method body is never edited.

In [29]:
import sys
import time
import logging
import functools
import pandas as pd

# Configure logging once so the decorators print to the notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,   # same stream as print() so output stays in order
    force=True,
)
logger = logging.getLogger("pipeline")

## Step 1 — Create the two decorators

- `@log_action` → logs when a method **starts** and **finishes** (and logs errors if it fails).
- `@log_time` → measures and prints the **execution time**.

`functools.wraps` keeps the original function's name and docstring, so `func.__name__` still says `read_csv` instead of `wrapper`.

In [30]:
def log_action(func):
    """Log when a function/method starts and finishes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"▶ START  {func.__qualname__}")
        try:
            result = func(*args, **kwargs)
        except Exception as e:
            logger.error(f"✖ FAILED {func.__qualname__}: {e}")
            raise
        logger.info(f"✔ FINISH {func.__qualname__}")
        return result
    return wrapper


def log_time(func):
    """Measure and print how long a function/method takes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱ {func.__qualname__} took {elapsed * 1000:.3f} ms")
        return result
    return wrapper

In [31]:
@log_action
@log_time
def slow_add(a, b):
    time.sleep(0.2)
    return a + b

print("Result:", slow_add(2, 3))
print("Name preserved by functools.wraps:", slow_add.__name__)

10:29:24 | INFO | ▶ START  slow_add
⏱ slow_add took 200.620 ms
10:29:24 | INFO | ✔ FINISH slow_add
Result: 5
Name preserved by functools.wraps: slow_add


## Step 2 — Apply the decorators to `CSVReader` and `DataCleaner`

The class logic is exactly the same as before — the only change is the decorator lines above each method. That is the Open/Closed Principle in action.

In [32]:
class CSVReader:
    """Responsible only for loading CSV data."""

    def __init__(self, file_path):
        self.file_path = file_path

    @log_action
    @log_time
    def read_csv(self):
        df = pd.read_csv(self.file_path)
        print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
        return df


class DataCleaner:
    """Responsible only for cleaning data."""

    def __init__(self, df):
        self.df = df.copy()

    @log_action
    @log_time
    def show_missing(self):
        missing = self.df.isnull().sum()
        print("Missing values per column:")
        print(missing[missing > 0])
        return missing

    @log_action
    @log_time
    def fill_missing_with_mean(self):
        num_cols = self.df.select_dtypes(include="number").columns
        self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].mean().round(1))
        return self.df

    @log_action
    @log_time
    def remove_duplicates(self):
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        print(f"Removed {before - len(self.df)} duplicate rows")
        return self.df

## Step 3 — Run the pipeline and watch the logs

In [33]:
reader = CSVReader("D:\I5\ADV\sample_data (2).csv")
raw_df = reader.read_csv()
raw_df

10:31:55 | INFO | ▶ START  CSVReader.read_csv
Loaded 7 rows × 7 columns
⏱ CSVReader.read_csv took 3.765 ms
10:31:55 | INFO | ✔ FINISH CSVReader.read_csv


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [34]:
cleaner = DataCleaner(raw_df)
cleaner.show_missing()
cleaner.fill_missing_with_mean()
clean_df = cleaner.remove_duplicates()
clean_df

10:32:18 | INFO | ▶ START  DataCleaner.show_missing
Missing values per column:
age          3
height_cm    1
weight_kg    1
score        2
dtype: int64
⏱ DataCleaner.show_missing took 9.174 ms
10:32:18 | INFO | ✔ FINISH DataCleaner.show_missing
10:32:18 | INFO | ▶ START  DataCleaner.fill_missing_with_mean
⏱ DataCleaner.fill_missing_with_mean took 15.119 ms
10:32:18 | INFO | ✔ FINISH DataCleaner.fill_missing_with_mean
10:32:18 | INFO | ▶ START  DataCleaner.remove_duplicates
Removed 0 duplicate rows
⏱ DataCleaner.remove_duplicates took 17.088 ms
10:32:18 | INFO | ✔ FINISH DataCleaner.remove_duplicates


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,30.2,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,86.0
3,4,David,30.2,167.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.2,162.0,60.0,Chicago,92.0


In [35]:
print("Missing values after cleaning:", clean_df.isnull().sum().sum())

Missing values after cleaning: 0


## Step 4 — Open/Closed Principle without touching the class

We can even add timing/logging to a class **we are not allowed to edit** by wrapping its method from the outside:

In [36]:
class Reporter:
    def summary(self, df):
        return df.describe().round(2)

# Extend behaviour from the outside — Reporter's source is unchanged
Reporter.summary = log_action(log_time(Reporter.summary))

Reporter().summary(clean_df)

10:33:25 | INFO | ▶ START  Reporter.summary
⏱ Reporter.summary took 39.439 ms
10:33:25 | INFO | ✔ FINISH Reporter.summary


,id,age,height_cm,weight_kg,score
count,7.00,7.00,7.00,7.00,7.00
mean,4.00,30.23,167.00,70.00,86.00
std,2.16,2.41,5.29,10.47,5.45
min,1.00,27.00,160.00,55.00,75.00
25%,2.50,29.50,163.50,64.00,85.50
50%,4.00,30.20,167.00,70.00,86.00
75%,5.50,30.20,170.00,76.00,89.00
max,7.00,35.00,175.00,85.00,92.00


## Summary

| Decorator | What it adds | Applied to |
|---|---|---|
| `@log_action` | Start / finish (and error) log messages | `read_csv`, `show_missing`, `fill_missing_with_mean`, `remove_duplicates` |
| `@log_time` | Execution time in ms | same methods |

**Why this follows Open/Closed:** logging and timing were added without modifying the bodies of `CSVReader` or `DataCleaner`. The decorators are reusable on any function, and removing them is as easy as deleting one line.

**Stacking order:** with `@log_action` on top of `@log_time`, `log_action` is the outer wrapper — its START prints first and FINISH prints last, and the timer measures only the method itself.

# Exercise 4: Factory Pattern for Data Transformations

The factory creates transformation objects from names, while every transformation follows the same `DataTransform` interface.

In [37]:
import pandas as pd
from abc import ABC, abstractmethod

In [38]:
class DataTransform(ABC):
    @abstractmethod
    # Purpose: Forces subclasses to implement a method. Prevents instantiation of the base class.
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a transformed copy of the input DataFrame."""
        raise NotImplementedError


class NormalizeColumns(DataTransform):
    """Normalize numeric columns to the range [0, 1]."""

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.select_dtypes(include='number').columns:
            minimum = result[column].min()
            maximum = result[column].max()
            if pd.notna(minimum) and pd.notna(maximum) and maximum != minimum:
                result[column] = (result[column] - minimum) / (maximum - minimum)
            elif pd.notna(minimum):
                result[column] = 0.0
        return result


class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    # text column: applies strip().lower() to string values
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.select_dtypes(include=['object', 'string']).columns:
            result[column] = result[column].map(
                lambda value: value.strip().lower() if isinstance(value, str) else value
            )
        return result


class TransformFactory:
    #One registry maps string names to classes, letting you create transformations dynamically without hardcoding imports or if/else chains.
    _TRANSFORMS = {
        'normalize': NormalizeColumns,
        'remove_duplicates': RemoveDuplicates,
        'duplicates': RemoveDuplicates,
        'standardize_text': StandardizeText,
        'text': StandardizeText,
    }

    @classmethod
    # Purpose: Factory method to create instances of DataTransform subclasses based on a string name.
    def create(cls, transform_name: str) -> DataTransform:
        key = transform_name.strip().lower()
        try:
            return cls._TRANSFORMS[key]()
        # Handle the case where the provided transform name is not in the registry.
        except KeyError as error:
            supported = ', '.join(sorted(cls._TRANSFORMS))
            raise ValueError(
            # Provide a helpful error message with the list of supported transformation names.
                f"Unknown transformation '{transform_name}'. Supported names: {supported}"
            ) from error

In [40]:
# Use the fully cleaned DataFrame produced by Exercise 2 (FillMean strategy).
cleaned_input = cleaned_df.copy()
assert cleaned_input.isna().sum().sum() == 0

normalized_df = TransformFactory.create('normalize').apply(cleaned_input)
deduplicated_df = TransformFactory.create('remove_duplicates').apply(cleaned_input)
standardized_df = TransformFactory.create('text').apply(cleaned_input)

print('Normalized numeric columns:')
display(normalized_df)

print('Duplicate rows removed:')
display(deduplicated_df)

print('Text columns standardized:')
display(standardized_df)

# Transformations can also be composed into a pipeline.
pipeline = ['text', 'normalize', 'remove_duplicates']
transformed_df = cleaned_input.copy()
for transform_name in pipeline:
    transformed_df = TransformFactory.create(transform_name).apply(transformed_df)

print('Composed pipeline result:')
display(transformed_df)

Normalized numeric columns:


,id,name,age,height_cm,weight_kg,city,score
0,0.000000,Alice,0.250,0.333333,0.433333,New York,0.588235
1,0.166667,Bob,0.000,0.800000,0.000000,Los Angeles,0.882353
2,0.333333,Charlie,1.000,0.533333,0.566667,Chicago,0.000000
3,0.500000,David,0.000,0.000000,0.833333,Houston,0.000000
4,0.666667,Eva,0.000,0.000000,0.000000,New York,0.764706
5,0.833333,Frank,0.375,1.000000,1.000000,Los Angeles,0.000000
6,1.000000,Grace,0.000,0.133333,0.166667,Chicago,1.000000


Duplicate rows removed:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0


Text columns standardized:


,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.0,165.0,68.0,new york,85.0
1,2,bob,27.0,172.0,55.0,los angeles,90.0
2,3,charlie,35.0,168.0,72.0,chicago,75.0
3,4,david,27.0,160.0,80.0,houston,75.0
4,5,eva,27.0,160.0,55.0,new york,88.0
5,6,frank,30.0,175.0,85.0,los angeles,75.0
6,7,grace,27.0,162.0,60.0,chicago,92.0


Composed pipeline result:


,id,name,age,height_cm,weight_kg,city,score
0,0.000000,alice,0.250,0.333333,0.433333,new york,0.588235
1,0.166667,bob,0.000,0.800000,0.000000,los angeles,0.882353
2,0.333333,charlie,1.000,0.533333,0.566667,chicago,0.000000
3,0.500000,david,0.000,0.000000,0.833333,houston,0.000000
4,0.666667,eva,0.000,0.000000,0.000000,new york,0.764706
5,0.833333,frank,0.375,1.000000,1.000000,los angeles,0.000000
6,1.000000,grace,0.000,0.133333,0.166667,chicago,1.000000


## Exercise 5 · A complete CSV cleaning pipeline

Build a reusable workflow with four design patterns:

| Pattern | Responsibility |
| :--- | :--- |
| **Template Method** | `DataPipeline.run()` fixes the order: load → clean → transform → save. |
| **Strategy** | `DataCleaner` delegates missing-value handling to an interchangeable strategy. |
| **Factory** | `TransformFactory` creates transformations from configuration names. |
| **Decorator** | `@log_action` and `@log_time` provide optional execution diagnostics. |

Run all cells in order. The example writes `dirty_sample.csv` and `cleaned_sample.csv`
in the notebook's working directory. Verbose diagnostics are disabled by default.

In [41]:
import os

print(os.getcwd())
print(os.listdir())

d:\I5\ADV
['cleaned_data.csv', 'sample_data (2).csv', 'TP1', 'TP1.ipynb']


In [43]:

import pandas as pd
from abc import ABC, abstractmethod
from pathlib import Path


# Step 1: Create the abstract class
class DataPipeline(ABC):

    def run(self):
        """Define the fixed workflow."""
        self.load()
        self.clean()
        self.transform()
        self.save()

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass


# Step 2: Implement the CSV pipeline
class CSVDataPipeline(DataPipeline):

    def __init__(self, input_file, output_file):
        self.input_file = input_file
        self.output_file = output_file
        self.data = None

    # Load the CSV file
    def load(self):
        if not Path(self.input_file).exists():
            raise FileNotFoundError(
                f"Cannot find file: {self.input_file}. "
                "Please check the file path."
            )

        self.data = pd.read_csv(self.input_file)
        print("1. Data loaded successfully")
        print(self.data)

    # Clean the data
    def clean(self):
        # Remove duplicate rows
        self.data = self.data.drop_duplicates()

        # Remove rows containing missing values
        self.data = self.data.dropna()

        # Remove unnecessary spaces from text values
        text_columns = self.data.select_dtypes(
            include="object"
        ).columns

        for column in text_columns:
            self.data[column] = self.data[column].str.strip()

        print("\n2. Data cleaned successfully")
        print(self.data)

    # Transform the data
    def transform(self):
        # Standardize column names
        self.data.columns = [
            column.strip().lower().replace(" ", "_")
            for column in self.data.columns
        ]

        print("\n3. Data transformed successfully")
        print(self.data)

    # Save the cleaned data
    def save(self):
        self.data.to_csv(self.output_file, index=False)
        print(f"\n4. Data saved successfully to {self.output_file}")


# Step 3: Create and run the pipeline
if __name__ == "__main__":
    pipeline = CSVDataPipeline(
        "sample_data (2).csv",
        "cleaned_data.csv"
    )

    pipeline.run()


1. Data loaded successfully
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0

2. Data cleaned successfully
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0

3. Data transformed successfully
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0

4. Data saved successfully to cleaned_data.csv


C:\Users\Dell\AppData\Local\Temp\ipykernel_16116\4072841995.py:62: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = self.data.select_dtypes(
